# Technical Challenge — Sourcing Analysis with AI

**Goal:** Analyze a sourcing/recruitment dataset to identify the best-performing channels, advancement signals, and when to persist or drop a candidate.

**AI Usage:** Logistic Regression predictive model to estimate hire probability per candidate.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Imports OK')

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv('../data/mock_sourcing_dataset_clean.csv')
bool_cols = ['response_received','screening_pass','interview1_pass','test_taken','offer_sent','hired']
for c in bool_cols:
    df[c] = df[c].map({True:True, False:False, 'True':True, 'False':False})
num_cols = ['response_time_days','technical_test_score','behavior_score','manager_score','stage_duration_days','years_experience']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print(f'Shape: {df.shape}')
print(f'Total hired: {df["hired"].fillna(False).sum()}')

## 3. Funnel Analysis

In [ ]:
funnel = pd.DataFrame({
    'stage': ['Sourced','Response','Screening','Interview 1','Assessment','Offer','Hired'],
    'count': [
        len(df),
        df['response_received'].fillna(False).sum(),
        df['screening_pass'].fillna(False).sum(),
        df['interview1_pass'].fillna(False).sum(),
        df['test_taken'].fillna(False).sum(),
        df['offer_sent'].fillna(False).sum(),
        df['hired'].fillna(False).sum()
    ]
})
funnel['pct_total'] = (funnel['count'] / funnel['count'].iloc[0] * 100).round(1)
funnel['drop_pct'] = funnel['count'].pct_change().mul(100).round(1)
print(funnel.to_string(index=False))

## 4. Channel Analysis

In [ ]:
channel = df.groupby('source_channel').agg(
    candidates=('candidate_id','count'),
    response_rate=('response_received','mean'),
    screening_rate=('screening_pass','mean'),
    interview_rate=('interview1_pass','mean'),
    assessment_rate=('test_taken','mean'),
    offer_rate=('offer_sent','mean'),
    hire_rate=('hired','mean'),
    avg_response_days=('response_time_days','mean'),
    avg_tech_score=('technical_test_score','mean')
).reset_index().sort_values('hire_rate', ascending=False)
pct_cols = ['response_rate','screening_rate','interview_rate','assessment_rate','offer_rate','hire_rate']
channel_display = channel.copy()
for c in pct_cols:
    channel_display[c] = (channel_display[c]*100).round(1).astype(str) + '%'
print(channel_display.to_string(index=False))

## 5. Advancement Signals — Scores & Responsiveness

In [ ]:
resp_time = df.groupby(df['hired'].fillna(False))['response_time_days'].agg(['mean','median','count'])
resp_time.index = ['Not hired','Hired']
print('Average response time (days) by outcome:')
print(resp_time)
print()
scores = df.groupby(df['hired'].fillna(False))[['technical_test_score','behavior_score','manager_score']].mean()
scores.index = ['Not hired','Hired']
print('Average scores by outcome:')
print(scores)

## 6. Rejection Reasons

In [ ]:
rejection = df[df['rejection_reason'].notna()]['rejection_reason'].value_counts()
print('Most common rejection reasons:')
print(rejection)

## 7. Recruiter Analysis

In [ ]:
recruiter = df.groupby('recruiter').agg(
    candidates=('candidate_id','count'),
    hire_rate=('hired','mean'),
    response_rate=('response_received','mean'),
    screening_rate=('screening_pass','mean'),
    avg_response_days=('response_time_days','mean')
).reset_index().sort_values('hire_rate', ascending=False)
for c in ['hire_rate','response_rate','screening_rate']:
    recruiter[c] = (recruiter[c]*100).round(1).astype(str) + '%'
print(recruiter.to_string(index=False))

## 8. AI — Predictive Model: Hire Probability

Logistic Regression model for pipeline prioritization support.
**Goal:** help recruiters focus on candidates with the highest conversion probability — not to replace human judgment.

In [ ]:
X = pd.get_dummies(df[['source_channel','department','work_mode','seniority','location']], drop_first=True)
X['response_time_days'] = df['response_time_days'].fillna(df['response_time_days'].median())
X['years_experience'] = df['years_experience'].fillna(df['years_experience'].median())
y = df['hired'].fillna(False).astype(int)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X, y)
auc = roc_auc_score(y, model.predict_proba(X)[:,1])
print(f'Model AUC: {auc:.3f}')
print('AUC > 0.65 is useful for pipeline prioritization')

In [ ]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
print('Top 15 variables most associated with hire outcome:')
print(coef.head(15).to_string())
print()
print('(+) increases hire probability / (-) reduces hire probability')

In [ ]:
df['hire_probability'] = model.predict_proba(X)[:,1]
df['priority'] = pd.cut(df['hire_probability'], bins=[0,0.05,0.12,1.0], labels=['Low','Medium','High'])
priority_summary = df.groupby('priority', observed=True).agg(
    total=('candidate_id','count'),
    hired_count=('hired', lambda x: x.fillna(False).sum()),
    actual_hire_rate=('hired', lambda x: x.fillna(False).mean())
).reset_index()
priority_summary['actual_hire_rate'] = (priority_summary['actual_hire_rate']*100).round(1).astype(str) + '%'
print('Candidate distribution by priority tier:')
print(priority_summary.to_string(index=False))

## 9. Practical Rules for Recruiters

**When to persist:**
- Candidate responded + passed screening + consistent scores
- High-conversion channel (GitHub, Inbound, Hunting)
- Stall reason is operational (timing, scheduling), not a fit issue

**When to stop:**
- No response after standard outreach sequence
- Multiple weak signals: low responsiveness + poor scores + low-conversion channel
- Confirmed structural reasons (salary mismatch, offer accepted elsewhere)

**Human-in-the-loop AI workflow:**
1. Model generates hire probability score per candidate
2. Recruiter reviews high-priority queue and makes final call
3. Mid-priority: nurturing with scheduled follow-up
4. Low-priority: archive or future re-engagement
5. Team recalibrates model monthly with new data

## 10. Conclusion

Three main levers identified:

1. **Channel mix**: GitHub, Inbound, and Hunting deliver the best final hire rates
2. **Engagement speed**: faster response time correlates with greater funnel advancement
3. **Disciplined prioritization**: using AI scoring concentrates recruiter effort where conversion probability is highest

The biggest funnel bottleneck sits between sourcing and response, and again between assessment and offer — both represent clear opportunities for operational improvement.